# Cómo funciona cada pieza y qué resultado dio

Este cuaderno explica, en palabras simples, **cada neurona, capa y arquitectura** de los cuadernos 01 a 08 que pasó por el banco de evaluación (`laboratorio/banco/`):

1. **Cómo funciona** (qué calcula, sin fórmulas largas).
2. **Qué resultado dio** contra la red estándar y contra la tecnología de la industria.
3. **Si vale la pena invertir** en ella.

También explica **por qué el tamaño (los parámetros) cambia el resultado**, con un experimento que puedes correr aquí mismo.

Las cifras salen de 5 semillas por pieza, medidas en datos de **prueba** que la red nunca vio al entrenar. Los informes completos están en `laboratorio/banco/resultados-*.md`.

> Todo este cuaderno corre en CPU (no necesita GPU). Para las competencias con datos reales usa los cuadernos 10 y 11.

## 1. Preparar

El repo es privado. Guarda un token de GitHub de **solo lectura** en *Secretos de Colab* (ícono de llave a la izquierda) con el nombre `GITHUB_TOKEN`. **Nunca lo pegues en una celda.**

In [ ]:
import os, subprocess
from google.colab import userdata

if not os.path.exists("/content/Notbook"):
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/SiriusMagic/Notbook.git", "/content/Notbook"], check=True)
%cd /content/Notbook
!pip install -q -r laboratorio/requirements.txt

## 2. Cómo se mide (para leer bien los resultados)

- **La misma red para todos**: 2 capas ocultas de 32 neuronas y una salida. Solo cambia la pieza que se prueba.
- **5 semillas**: cada semilla es un arranque distinto al azar. Si una pieza gana con una semilla y pierde con otra, fue suerte.
- **Datos de prueba**: se mide con datos que la red no vio. Acertar al entrenar es fácil; lo que importa es acertar con datos nuevos.
- **Veredicto**: 🟢 mejor, ⚪ igual (la diferencia cabe en el ruido), 🔴 peor. Se usa una prueba estadística por pares y una corrección para no contar ganadores por azar cuando se comparan muchas piezas a la vez.
- **Rivales de la industria**: GELU, SiLU, Mish, Dropout, BatchNorm, LayerNorm, conexiones residuales, CNN, LSTM, Transformer, mezcla de expertos, Random Forest, Gradient Boosting… Ganarle a la red estándar (ReLU) no basta: para invertir, una pieza tiene que ganarle a lo que se usa hoy.

### Las tareas

| Tarea | Qué es | Métrica |
|---|---|---|
| `regresion_cubica` | aprender la curva y = x³ − x con ruido | error (MSE): menos es mejor |
| `circulos` | separar dos círculos, uno dentro del otro | acierto |
| `lunas` | separar dos medias lunas | acierto |
| `xor` | puntos de los 4 cuadrantes (los opuestos son de la misma clase) | acierto |
| `serie_seno` | predecir el siguiente valor de una onda | error (MSE) |
| `digitos` | reconocer dígitos escritos a mano (imágenes 8×8) | acierto |
| `caotica_20d` | 4 clases definidas por reglas enredadas en 20 números (del cuaderno 08) | acierto |

## 3. Por qué el tamaño cambia el resultado

Una red con **más parámetros** (más pesos que aprender) puede representar funciones más complicadas. Eso tiene dos caras:

- **Ayuda** cuando la tarea es difícil y la red se queda corta.
- **Estorba** cuando la red empieza a **memorizar**: aprende de memoria los ejemplos de entrenamiento (incluido su ruido) en vez de la regla general. Se nota porque acierta casi 100 % al entrenar y mucho menos en la prueba.

Por eso, cuando una pieza nueva «gana», lo primero es preguntar: **¿gana por la idea o por tener más parámetros?** La comparación justa es contra una red estándar **del mismo tamaño**.

### Barrido de tamaño (5 semillas por fila)

**Tarea caótica 20D** (la red memoriza mucho):

| Neuronas por capa | Parámetros | Red estándar: prueba | Red estándar: entreno | Con Dropout: prueba |
|---|---|---|---|---|
| 8 | 276 | 75,6 % ± 1,4 | 81,4 % | 75,3 % ± 0,5 |
| 16 | 676 | 72,8 % ± 0,7 | 87,2 % | 76,5 % ± 0,7 |
| 32 | 1 860 | 70,0 % ± 0,9 | 98,4 % | 77,4 % ± 1,2 |
| 64 | 5 764 | 71,3 % ± 1,4 | 100,0 % | 75,1 % ± 0,7 |
| 128 | 19 716 | 71,5 % ± 0,8 | 100,0 % | 75,5 % ± 0,9 |

**Círculos** (tarea sencilla con ruido):

| Neuronas por capa | Parámetros | Red estándar: prueba | Red estándar: entreno | Con Dropout: prueba |
|---|---|---|---|---|
| 8 | 114 | 85,7 % ± 1,4 | 83,5 % | 83,7 % ± 1,7 |
| 16 | 354 | 85,3 % ± 0,9 | 84,1 % | 85,7 % ± 0,5 |
| 32 | 1 218 | 84,8 % ± 0,6 | 84,4 % | 85,2 % ± 1,0 |
| 64 | 4 482 | 84,3 % ± 1,0 | 85,2 % | 83,9 % ± 1,1 |
| 128 | 17 154 | 82,0 % ± 1,1 | 86,4 % | 82,8 % ± 1,0 |

**Dígitos 8×8**:

| Neuronas por capa | Parámetros | Red estándar: prueba | Red estándar: entreno | Con Dropout: prueba |
|---|---|---|---|---|
| 8 | 682 | 93,7 % ± 1,1 | 99,8 % | 93,3 % ± 0,3 |
| 16 | 1 482 | 96,9 % ± 0,7 | 100,0 % | 96,6 % ± 0,3 |
| 32 | 3 466 | 97,1 % ± 0,5 | 100,0 % | 97,1 % ± 0,5 |
| 64 | 8 970 | 97,0 % ± 0,2 | 100,0 % | 97,8 % ± 0,2 |
| 128 | 26 122 | 96,7 % ± 0,6 | 100,0 % | 96,9 % ± 0,7 |

### Qué se aprende de estas tablas

- **Caótica 20D: más grande es peor.** Con 8 neuronas la red estándar acierta 75,6 %; con 32, solo 70,0 %. La red de 32 acierta 98 % al entrenar: memoriza. A partir de 64 neuronas sube un poco (71-72 %), pero nunca vuelve al nivel de la red chica.
- **Dropout ayuda justo donde la red memoriza** (32 neuronas: de 70,0 % a 77,4 %). Con 8 neuronas casi no cambia nada, porque esa red no tiene con qué memorizar.
- **Círculos**: la tarea es tan sencilla que 8 neuronas bastan (85,7 %). Con 128 baja a 82,0 %.
- **Dígitos**: aquí sí hacen falta neuronas. Con 8 acierta 93,7 % y con 16 a 64, alrededor de 97 %. Dropout con 64 da lo mejor (97,8 %).
- **Consecuencia para tus piezas**: el banco usa 32 neuronas, y en la tarea caótica ese tamaño ya memoriza. Por eso ahí ganan las piezas que frenan la memorización (Dropout, la dendrítica, DMNA, la genética invertida). **Pero una red estándar más chica logra casi lo mismo (75,6 %) sin ninguna idea nueva.** Para invertir en una pieza, tiene que ganarle también a la red estándar de su mejor tamaño.

### Ejemplo real: el cerebro modular del cuaderno 06

En la tarea caótica, el cerebro modular (CorticalMoE) llegó a 73,8 % contra 70,0 % de la red estándar: parecía una victoria. Pero tiene **23 428 parámetros contra 1 860** (12 veces más). Contra una red estándar de su mismo tamaño (72,3 %) gana solo 1,5 puntos, y la red chica con Dropout llega a 77,4 %.

### Pruébalo tú

La celda siguiente repite el barrido. Cambia `TAREA`, `PIEZAS` o `TAMANOS` para probar otras combinaciones.

In [ ]:
import statistics as st
import matplotlib.pyplot as plt
from laboratorio.banco.evaluar import Config, correr

TAREA = "caotica_20d"
PIEZAS = ["estandar", "dropout"]
TAMANOS = [8, 16, 32, 64, 128]
SEMILLAS = (0, 1, 2)          # 5 semillas da cifras más firmes, pero tarda más

fig, ax = plt.subplots(figsize=(7, 4))
for pieza in PIEZAS:
    params, prueba, entreno = [], [], []
    for n in TAMANOS:
        corridas = correr(pieza, TAREA, config=Config(semillas=SEMILLAS, medir=False, ocultas=n))
        params.append(corridas[0].parametros)
        prueba.append(st.mean(c.valor for c in corridas))
        entreno.append(st.mean(c.valor_entreno for c in corridas))
        print(f"{pieza:10s} {n:4d} neuronas  {params[-1]:7,d} parámetros  prueba {prueba[-1]:.3f}  entreno {entreno[-1]:.3f}")
    ax.plot(params, prueba, "o-", label=f"{pieza} (prueba)")
    ax.plot(params, entreno, "o--", alpha=0.5, label=f"{pieza} (entreno)")
ax.set_xscale("log"); ax.set_xlabel("parámetros"); ax.set_ylabel("métrica"); ax.set_title(TAREA); ax.legend()
plt.show()

## 4. Mirar una pieza por dentro

Escribe el nombre de cualquier pieza (la lista completa sale al correr la celda) y verás su explicación, cuántos parámetros tiene y cómo se arma por dentro.

In [ ]:
import inspect, torch
from laboratorio.banco.piezas import PIEZAS, Red

print("Piezas disponibles:", ", ".join(PIEZAS))

NOMBRE = "cerebro_moe"   # cámbialo

pieza = PIEZAS[NOMBRE]
red = Red(pieza, entradas=20, salidas=4, ocultas=32)
interna = getattr(red, "topologia", None) or red.capa1
print(f"\n{NOMBRE}: {pieza.descripcion}  (de: {pieza.origen})")
print(f"Parámetros con 20 entradas, 4 salidas y 32 neuronas: {sum(p.numel() for p in red.parameters()):,}")
print("\nCómo funciona (del código):\n")
print(inspect.getdoc(type(interna)) or "(sin explicación)")
print("\nEstructura:\n", red)

## 5. ¿Aprende o memoriza?

Compara una pieza con la red estándar **del mismo tamaño** y muestra el acierto al entrenar y en la prueba. Si el acierto al entrenar es mucho mayor que el de prueba, la red memoriza.

In [ ]:
from laboratorio.banco.evaluar import Config, correr
from laboratorio.banco.piezas import PIEZAS, Red
from laboratorio.banco.tareas import cargar_tarea

def comparar_justo(pieza, tarea="caotica_20d", semillas=(0, 1, 2)):
    """Busca el tamaño de red estándar con parámetros parecidos a la pieza y compara las dos."""
    p = correr(pieza, tarea, config=Config(semillas=semillas, medir=False))
    objetivo = p[0].parametros
    t = cargar_tarea(tarea)
    tamano = lambda n: sum(q.numel() for q in Red(PIEZAS["estandar"], t.entradas, t.salidas, n).parameters())
    mejor = min(range(4, 257), key=lambda n: abs(tamano(n) - objetivo))
    e = correr("estandar", tarea, config=Config(semillas=semillas, medir=False, ocultas=mejor))
    for nombre, cs in ((pieza, p), (f"estándar de {mejor} neuronas", e)):
        prueba = sum(c.valor for c in cs) / len(cs); entreno = sum(c.valor_entreno for c in cs) / len(cs)
        print(f"{nombre:32s} {cs[0].parametros:7,d} parámetros   prueba {prueba:.3f}   entreno {entreno:.3f}")

comparar_justo("biomas")

## 6. Cuaderno 01: texto y lenguaje

Estas piezas se miden con datos reales de texto en el cuaderno 11 (necesita GPU).

| Pieza | Cómo funciona | Resultado |
|---|---|---|
| Tokenizador por sílabas | corta las palabras en sílabas con reglas del español; cada sílaba es una «pieza» del vocabulario | compite contra BPE (GPT, Llama), Unigram (T5, Gemma) y el de GPT-4o con el mismo vocabulario de 8 000 unidades; se mide en el cuaderno 11 |
| Posición jerárquica | en vez de numerar cada token, dice «palabra n.º X, sílaba n.º Y» | compite contra la posición aprendida y la senoidal de los Transformers (cuaderno 11) |
| Segmentador de sílabas (BiLSTM) | una red lee la palabra letra por letra en los dos sentidos y marca dónde corta | comparado con reglas, pyphen y un Transformer de letras, con 128 palabras silabeadas a mano según la RAE |
| Corrector GNN | cada letra es un nodo de un grafo; los nodos se pasan mensajes con sus vecinas y deciden la letra correcta | **sin conexiones residuales olvida la letra original y empeora el texto**; con ellas lo mejora con muy pocos parámetros |
| Atención binaria con RL | un agente decide qué palabras leer (sí/no) y aprende con recompensa | compite contra TextCNN, Transformer y fastText; en el cuaderno el clasificador no se entrenaba (arreglado) |
| Atención por niveles con RL | igual, pero el agente elige cuánta atención dar (varios niveles) | ídem |
| Atención «hebbiana» | las palabras que aparecen juntas refuerzan su conexión | ídem |

## 7. Cuaderno 02: capas y activaciones

**Activaciones y neuronas** (primera tanda):

| Pieza | Cómo funciona | Resultado |
|---|---|---|
| PReLU adaptativa | ReLU cuya pendiente para negativos se aprende | ⚪ empata |
| Integración no lineal | cada neurona combina sus entradas con términos no lineales antes de sumar | ⚪ empata en tareas pequeñas con 9× parámetros; 🟢 75,7 % en la caótica (pero con 9× parámetros) |
| Activación adaptativa | la forma de la activación tiene parámetros que se aprenden | ⚪ empata |
| IDA | activación que mezcla varias curvas | 🟢 en la serie de seno (≈ 7 % menos error), pero ahí una regresión lineal gana a todas |
| SMA | la activación muta al azar durante el entrenamiento | ⚪ empata |
| Umbral dinámico | el umbral de cada neurona se ajusta según su actividad | ⚪ empata |
| RMB | capa con memoria de lotes anteriores | inestable: algunas semillas se disparan |
| HMN | memoria híbrida | 🔴 peor en todo: su memoria se ata a la posición de cada fila y memoriza |
| Seno (del 04) | sin(x) como activación | 🟢 en la serie de seno (misma advertencia que IDA); 🔴 en la caótica |

**Capas con memoria e inicializadores** (segunda tanda):

| Pieza | Cómo funciona | Resultado |
|---|---|---|
| Memoria de errores | recuerda las muestras con más error y las vuelve a corregir | 🔴 peor en regresión: su paso extra estorba a Adam |
| Memoria de gradiente | paso extra con el promedio de gradientes recientes | ⚪ empata: es un *momentum*, y Adam ya lo tiene |
| LR por neurona | cada peso tiene su propia tasa de aprendizaje | 🔴 peor en regresión y XOR (es un RMSprop casero) |
| Memoria con atención | mezcla gradientes recordados según el parecido de la entrada | ⚪ empata |
| Memoria con embeddings | guarda las pérdidas como vectores y ajusta la capa | ⚪ empata con 3-8× parámetros |
| Inicio Perlin / Lorenz | los pesos iniciales salen de ruido Perlin o del atractor de Lorenz | ⚪ empatan: con Adam el punto de partida casi no importa |
| Denso-cruzada / densa de entrada | cada capa recibe todas las anteriores (y la entrada) | ⚪ empatan con 3-8× parámetros |

**Neuronas especiales** (tercera y cuarta tanda):

| Pieza | Cómo funciona | Resultado |
|---|---|---|
| Polinómica | agrega x², x·y… de las entradas | 🔴 memoriza (93 % al entrenar, 79 % en prueba en círculos) |
| Resonante | solo se activa si la entrada supera un umbral de «fuerza» | 🔴 el umbral la apaga y deja de aprender |
| DMNA | ruido al entrenar que depende de la pérdida y se apaga con el tiempo | ⚪ en tareas pequeñas; 🟢 72,2 % en la caótica |
| GMO | muta todos los pesos y acepta el cambio si mejora 1 % | ⚪ nunca aceptó una mutación |
| CDM | un extractor limpia la entrada antes de la red | ⚪ empata (el original usaba datos «limpios» que en la vida real no existen) |
| HiperReLU / CCN | ReLU ×10 y normalización con «turbulencia» | 🔴 multiplicar ×10 desestabiliza |
| Dendrítica | cada neurona tiene dos «ramas» (ReLU y exponencial) y un cuerpo sigmoide | 🔴 en tareas pequeñas; **🟢 76,1 % en la caótica, casi como Dropout (77,4 %) y con los mismos parámetros**; ojo: la red estándar de 8 neuronas llega a 75,6 % (sección 3) |
| Synaptic tagging | marca los pesos con gradiente fuerte y los refuerza cada 10 pasos | 🟢 en la serie, pero una tanh sola da lo mismo |
| Ruteo adaptativo | una compuerta reparte entre dos rutas | ⚪ empata |
| CNN con entrada densa | cada capa de la CNN recibe la imagen original | ⚪ empata con la CNN normal (97,7 % contra 98,4 %) |

## 8. Cuaderno 03: neuronas alternativas

| Pieza | Cómo funciona | Resultado |
|---|---|---|
| Neuronas KNN | cada neurona guarda 5 patrones y se activa según qué tan cerca está la entrada | 🔴 solo aprende la capa de salida |
| Genética (tal cual) | reemplaza las neuronas «más ajustadas» por copias mutadas de las menos ajustadas | 🔴 se derrumba: las menos ajustadas son las **muertas**, así copia neuronas muertas |
| Genética arreglada | igual, pero solo copia neuronas vivas | 🔴 va destruyendo la red poco a poco |
| Genética invertida | al revés: recicla las neuronas que no se usan copiando las más útiles (como ReDo, 2023) | ⚪ en tareas pequeñas; 🟢 72,2 % en la caótica |
| Árbol, Naive Bayes, SVM, Random Forest, Gradient Boosting | clasificadores clásicos completos | **Gradient Boosting: 94 % en la caótica**, muy por encima de cualquier red (77 %) |

## 9. Cuaderno 04: visión

| Pieza | Cómo funciona | Resultado |
|---|---|---|
| Decorrelación | castiga que dos neuronas de una capa respondan igual | ⚪ empata |
| CNN auto-decorrelada | lo mismo entre los filtros de una CNN | ⚪ empata con la CNN normal (97,9 % contra 98,4 %) |
| Papelito preguntón | dos capas se «preguntan» entre sí y se castiga que se parezcan | ⚪ / 🔴 en círculos; 5× parámetros y memoriza |
| Celda recurrente RCF | dos estados acoplados en cada paso de la serie | 🔴 peor que la LSTM |

## 10. Cuaderno 05: atención

| Pieza | Cómo funciona | Resultado |
|---|---|---|
| Transformer con máscara | cada cabeza aprende qué posiciones mirar con una máscara | ⚪ empata con el Transformer normal con casi el doble de parámetros. La versión original multiplica la máscara y no bloquea nada; la arreglada suma log(máscara) |
| AttentiveLinear | cada neurona reparte su salida con factores fijos | 🔴 se derrumba: los factores no dependen de la entrada, así que no es atención |
| Atención local | agrega el promedio de entradas pasadas pesado por 1/error | ⚪ empata; 🔴 en XOR |

## 11. Cuaderno 06: refuerzo y cerebro modular

| Pieza | Cómo funciona | Resultado |
|---|---|---|
| RlNeuron | cada neurona multiplica su salida por un factor que crece con el tamaño de su gradiente | ⚪ empata. Pese al nombre no es refuerzo: el factor solo crece |
| Votación de cabezas | una capa común y 4 «cabezas» que votan promediando sus respuestas | ⚪ empata (🟢 solo en la serie de seno) |
| Biomas | extractor → «memoria» LSTM → procesamiento | ⚪ empata; la LSTM recibe secuencias de un solo paso, así que no recuerda nada |
| Cerebro modular (CorticalMoE) | un «tálamo» elige 1 de 4 expertos (memoria, lógica, instinto, contexto) por ejemplo y aprende con recompensa | 🔴 en regresión: todos los ejemplos reciben la misma recompensa. En la caótica 73,8 %, pero con 12× parámetros (ver la sección 3) |
| Cerebro modular arreglado | recompensa propia para cada ejemplo | ⚪ empata |
| Tálamo top-2 | el router mezcla los 2 mejores expertos y ve la salida anterior | ⚪ empata; agregar la entropía no ayudó |
| Mezcla suave de expertos (industria) | los 4 expertos se mezclan con pesos que suman 1 | ⚪ empata |

## 12. Cuaderno 07: grafos

| Pieza | Cómo funciona | Resultado |
|---|---|---|
| Anti-gradiente | invierte el gradiente entre el extractor y el clasificador | 🔴 peor en 5 de 6 tareas. En la industria (DANN) va en una rama aparte, no en el camino principal |

## 13. Resumen: ¿dónde vale la pena invertir?

- **En tareas pequeñas casi nada gana** a la red estándar. Las mejoras que se veían en los cuadernos no se sostenían con varias semillas y datos de prueba.
- **Donde la red memoriza** (tarea caótica) sí ganan las piezas que la frenan: **Dropout (77,4 %)**, tu **neurona dendrítica (76,1 %)**, DMNA y la genética invertida (72,2 %). Pero una red estándar de solo 8 neuronas llega a 75,6 % (sección 3): la dendrítica gana poco frente a eso. Sigue siendo la pista más fuerte de tus piezas, y se debe probar con datos reales en tabla.
- **Más parámetros no es mejor idea**: varias «victorias» (cerebro modular, integración no lineal, biomas) se explican por el tamaño. Compáralas siempre con una red estándar del mismo tamaño (sección 5).
- **En datos de tabla, los árboles ganan**: Gradient Boosting llega a 94 % en la caótica.
- **Falta**: probar las pistas (dendrítica, DMNA, genética invertida, IDA) con datos reales y GPU (cuadernos 10 y 11), y la parte generativa (GAN, difusión de texto, HCMG…), que va al final.